In [95]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import yfinance as yf

In [96]:
data= yf.download ('BTC-USD',start='2018-01-01',end='2022-01-12',interval="1d")
data.columns= data.columns.droplevel(1)
data=data.reset_index()
data

[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open,Volume
0,2018-01-01,13657.200195,14112.200195,13154.700195,14112.200195,10291200000
1,2018-01-02,14982.099609,15444.599609,13163.599609,13625.000000,16846600192
2,2018-01-03,15201.000000,15572.799805,14844.500000,14978.200195,16871900160
3,2018-01-04,15599.200195,15739.700195,14522.200195,15270.700195,21783199744
4,2018-01-05,17429.500000,17705.199219,15202.799805,15477.200195,23840899072
...,...,...,...,...,...,...
1467,2022-01-07,41557.902344,43153.570312,41077.445312,43153.570312,84196607520
1468,2022-01-08,41733.941406,42228.941406,40672.277344,41561.464844,28066355845
1469,2022-01-09,41911.601562,42663.949219,41338.160156,41734.726562,21294384372
1470,2022-01-10,41821.261719,42199.484375,39796.570312,41910.230469,32104232331


**STRATEGY**

In [97]:
def strategy(data):
    EMA= data["Close"].ewm(span=7).mean()
    buy_band= EMA+ (data["High"]-data["Low"])
    sell_band= EMA - (data["High"]-data["Low"])
    i=0
    data["signal1"]=0
    while i<len(data):
      if data.loc[i,"Close"]>buy_band[i]:
        data.loc[i,"signal1"]=1
      elif data.loc[i,"Close"]<sell_band[i]:
        data.loc[i,"signal1"]=-1
      i+=1

    ema_fast= data["Close"].ewm(span=6).mean()
    ema_slow= data["Close"].ewm(span=13).mean()
    macd=ema_fast-ema_slow
    signal_line= macd.ewm(span=5).mean()
    data["signal2"]=0
    i=0
    while i<len(data):
      if macd[i]>signal_line[i]:
        data.loc[i,"signal2"]=1
      elif macd[i]<signal_line[i]:
        data.loc[i,"signal2"]=-1
      i+=1

    i=0
    data["hybrid"]=0
    while i<len(data):
      if (data.loc[i,"signal1"]==1 and data.loc[i,"signal2"]==1 ):
        data.loc[i,"hybrid"]=1
      elif (data.loc[i,"signal1"]==-1 and data.loc[i,"signal2"]==-1):
        data.loc[i,"hybrid"]=-1
      i+=1

    i=0
    while i < len(data) and data.loc[i,"hybrid"]!=1:  ### removing all sells before first buy.
      data.loc[i,"hybrid"]=0
      i+=1

    i=0
    while i<len(data)-1:            ### Keeping one trade active at a time....removing repetitive buy and sell...
      if data.loc[i,"hybrid"]==-1:
        j=i+1
        while j<len(data) and data.loc[j,"hybrid"]!=1:
          data.loc[j,"hybrid"]=0
          j+=1
      i+=1
    i=0
    while i<len(data)-1:
      if data.loc[i,"hybrid"]==1:
        j=i+1
        while j<len(data) and data.loc[j,"hybrid"]!=-1:
          data.loc[j,"hybrid"]=0
          j+=1
      i+=1
    return data
data=strategy(data)
data

Price,Date,Close,High,Low,Open,Volume,signal1,signal2,hybrid
0,2018-01-01,13657.200195,14112.200195,13154.700195,14112.200195,10291200000,0,0,0
1,2018-01-02,14982.099609,15444.599609,13163.599609,13625.000000,16846600192,0,1,0
2,2018-01-03,15201.000000,15572.799805,14844.500000,14978.200195,16871900160,0,1,0
3,2018-01-04,15599.200195,15739.700195,14522.200195,15270.700195,21783199744,0,1,0
4,2018-01-05,17429.500000,17705.199219,15202.799805,15477.200195,23840899072,0,1,0
...,...,...,...,...,...,...,...,...,...
1467,2022-01-07,41557.902344,43153.570312,41077.445312,43153.570312,84196607520,-1,-1,0
1468,2022-01-08,41733.941406,42228.941406,40672.277344,41561.464844,28066355845,-1,-1,0
1469,2022-01-09,41911.601562,42663.949219,41338.160156,41734.726562,21294384372,-1,-1,0
1470,2022-01-10,41821.261719,42199.484375,39796.570312,41910.230469,32104232331,0,-1,0


In [98]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=data["Date"], y=data["Close"], name = "Close", line = dict(color='blue', width=1.5)))
fig.add_trace(go.Scatter(x=data["Date"][data['hybrid']==1], y=data['Close'][data['hybrid']==1], name = "Buy Signal", mode = 'markers', marker = dict(symbol = 'triangle-up', color='green', size=8)))
fig.add_trace(go.Scatter(x=data["Date"][data['hybrid']==-1], y=data['Close'][data['hybrid']==-1], name = "Sell Signal", mode = 'markers', marker = dict(symbol = 'triangle-down', color='red', size=8)))
fig.update_layout(title=dict(text="SIGNALS",x=0.45,y=0.9,font=dict(size=30)),yaxis_title="Price",xaxis_title="Date")
fig.show()

# **WITH RISK MANAGEMENT**

**BACKTESTING WITH STOPLOSS** **(long+short)**

In [99]:
def backtest(data, stop_loss_pct):
    capital= 100000
    in_position= False
    position_type= None
    entry_price= 0
    quantity= 0
    stop_loss= 0
    entry_index= 0
    trades= []
    portfolio_values= []
    i=0
    while i<len(data):
        signal= data['hybrid'].iloc[i]
        price= data['Close'].iloc[i]

        if not in_position:
            if signal== 1:
                position_type= "long"
                entry_price= price
                quantity= int(capital / price)
                capital -= quantity*price
                stop_loss= entry_price*(1 - stop_loss_pct)
                entry_index= i
                in_position= True

            elif signal== -1:
                position_type= "short"
                entry_price= price
                quantity= int(capital / price)
                capital += quantity * price
                stop_loss= entry_price*(1 + stop_loss_pct)
                entry_index= i
                in_position= True

        else:
            if position_type== "long":
                if price<stop_loss or signal== -1:
                    capital += quantity*price
                    trades.append({
                        "entry":entry_index,
                        "exit":i,
                        "P and L":(price-entry_price)*quantity,
                        "type":"long",
                        "duration":i-entry_index
                    })
                    in_position= False
                    quantity=0
                    position_type = None

            elif position_type== "short":
                if price>stop_loss or signal== 1:
                    capital -= quantity*price
                    trades.append({
                        "entry":entry_index,
                        "exit":i,
                        "P and L":(entry_price-price)*quantity,
                        "type":"short",
                        "duration":i-entry_index
                    })
                    in_position= False
                    quantity=0
                    position_type = None

        if in_position:
          if position_type == "long":
            port_value = capital + quantity * price
          elif position_type == "short":
            profit = (entry_price - price) * quantity
            port_value = capital + profit
        else:
            port_value = capital

        portfolio_values.append(port_value)
        i+=1

    #DataFrame
    trade= pd.DataFrame(trades)
    data['portfolio value']= portfolio_values

    print("Total trades:",len(trade))
    print("Final capital:",capital)
    print("Win:",(trade['P and L']> 0).sum())
    print("Losse:",(trade['P and L']< 0).sum())
    print("Final Portfoilio value:",data['portfolio value'].iloc[len(data)-1])
    return trade
backtest=backtest(data,stop_loss_pct=0.01)
backtest

Total trades: 47
Final capital: 622199.3146972656
Win: 13
Losse: 34
Final Portfoilio value: 622199.3146972656


,entry,exit,P and L,type,duration
0,5,6,-5247.001953,long,1
1,30,45,492.292969,short,15
2,66,69,-1836.201172,short,3
3,102,130,6000.833008,long,28
4,198,212,3303.694824,long,14
5,236,248,-3510.300293,long,12
6,265,266,-1613.076172,long,1
7,284,287,-4829.399414,short,3
8,288,290,-1671.598633,long,2
9,302,308,-1195.461914,short,6


**METRICS (WITH STOPLOSS)**

In [100]:
def metrics(data,trade,capital):
    print("Returns:", (trade['P and L'].sum()/ capital) * 100,'%')

    number_of_stock= int(capital/data['Close'].iloc[0])
    final_value = capital-number_of_stock*(data['Close'].iloc[0] - data['Close'].iloc[-1])

    drawdown= []
    dip= []
    returns_for_sharpe= []
    i=0
    while i<len(trade):
        entry_index= trade['entry'][i]
        exit_index= trade['exit'][i]

        initial_value= data['portfolio value'].iloc[entry_index]
        period_values= data['portfolio value'].iloc[entry_index : exit_index+1]
        min_value= np.min(period_values)
        dip.append(( min_value-initial_value)*100/initial_value)

        excess_return= (trade['P and L'][i] / initial_value)
        returns_for_sharpe.append(excess_return)

        max_drawdown= 0
        temp= initial_value
        j= entry_index + 1
        while j< exit_index + 1:
            temp= max(temp, data['portfolio value'].iloc[j])
            drawdown_value = (temp- data['portfolio value'].iloc[j]) / temp
            max_drawdown = max(max_drawdown, drawdown_value)
            j+=1
        drawdown.append(100*max_drawdown)
        i+=1

    print("Benchmark Return (in amount):", final_value - capital)
    print("\nNumber of closed trades:", len(trade))
    print("\nMax holding time:", trade['duration'].max())
    print("Avg Holding time:", trade['duration'].mean())
    print("\nGross Profit:", trade['P and L'].sum())
    print("Net Profit:", trade['P and L'].sum() - 20 * len(trade))
    print("\nMax drawdown (in %):", np.max(drawdown))
    print("Avg drawdown (in %):", np.mean(drawdown))
    print("\nMax dip (in %):", np.min(dip))
    print("Avg dip (in %):", np.mean(dip))

    if np.std(returns_for_sharpe) != 0:
        sharpe = np.sqrt(252) * (np.mean(returns_for_sharpe) / np.std(returns_for_sharpe))
    else:
        sharpe = 0
    print("\nSharpe Ratio:", sharpe)
metrics(data,backtest,100000)

Returns: 522.1993146972657 %
Benchmark Return (in amount): 203550.5869140625

Number of closed trades: 47

Max holding time: 165
Avg Holding time: 14.595744680851064

Gross Profit: 522199.3146972656
Net Profit: 521259.3146972656

Max drawdown (in %): 56.45109196184394
Avg drawdown (in %): 27.11339193028979

Max dip (in %): -56.45109196184394
Avg dip (in %): -22.820189158129175

Sharpe Ratio: 2.630594314270871


**PORTFOLIO (WITH STOPLOSS)**

In [101]:
fig=go.Figure()
fig.add_trace(go.Scatter(x=data["Date"], y=data["portfolio value"], line = dict(color='blue', width=2)))
fig.update_layout(title=dict(text="Portfolio value(with stoploss)",x=0.45,y=0.9,font=dict(size=30)),yaxis_title="portfolio value",xaxis_title="Date")
fig.show()

# **WITHOUT RISK MANAGEMENT (FOR COMPARISION)**

**BACKTESTING (long+short)**

In [102]:
def capital(data):
    capital = 100000
    quantity = 0
    position = None
    entry_price = 0
    cap = []

    for i in range(len(data)):
        price = data['Close'].iloc[i]
        signal = data['hybrid'].iloc[i]

        # Entry signals
        if position is None:
            if signal == 1:
                quantity = int(capital/price)
                capital -= quantity * price
                entry_price = price
                position = 'long'
            elif signal == -1:
                quantity = int(capital/price)
                capital += quantity * price
                entry_price = price
                position = 'short'

        elif position == 'long':
            if signal == -1:
                capital += quantity * price  # close long
                quantity = int(capital/price)
                capital += quantity * price  # enter short
                entry_price = price
                position = 'short'
            else:
                value = capital + quantity * price
                cap.append(value)
                continue

        elif position == 'short':
            if signal == 1:
                capital -= quantity * price  # close short
                quantity = int(capital/price)
                capital -= quantity * price  # enter long
                entry_price = price
                position = 'long'
            else:
                profit = (entry_price - price) * quantity
                value = capital + profit
                cap.append(value)
                continue

        if position == 'long':
            value = capital + quantity * price
        elif position == 'short':
            profit = (entry_price - price) * quantity
            value = capital + profit
        else:
            value = capital
        cap.append(value)

    data['portfolio(without stoploss)'] = cap
    return data
capital(data)

Price,Date,Close,High,Low,Open,Volume,signal1,signal2,hybrid,portfolio value,portfolio(without stoploss)
0,2018-01-01,13657.200195,14112.200195,13154.700195,14112.200195,10291200000,0,0,0,100000.000000,100000.000000
1,2018-01-02,14982.099609,15444.599609,13163.599609,13625.000000,16846600192,0,1,0,100000.000000,100000.000000
2,2018-01-03,15201.000000,15572.799805,14844.500000,14978.200195,16871900160,0,1,0,100000.000000,100000.000000
3,2018-01-04,15599.200195,15739.700195,14522.200195,15270.700195,21783199744,0,1,0,100000.000000,100000.000000
4,2018-01-05,17429.500000,17705.199219,15202.799805,15477.200195,23840899072,0,1,0,100000.000000,100000.000000
...,...,...,...,...,...,...,...,...,...,...,...
1467,2022-01-07,41557.902344,43153.570312,41077.445312,43153.570312,84196607520,-1,-1,0,622199.314697,492778.266113
1468,2022-01-08,41733.941406,42228.941406,40672.277344,41561.464844,28066355845,-1,-1,0,622199.314697,491898.070801
1469,2022-01-09,41911.601562,42663.949219,41338.160156,41734.726562,21294384372,-1,-1,0,622199.314697,491009.770020
1470,2022-01-10,41821.261719,42199.484375,39796.570312,41910.230469,32104232331,0,-1,0,622199.314697,491461.469238


**TRADEWISE DATAFRAME**

In [103]:
def tradewise(data):  ### FROM LAST ASSIGNMENT
    trade_index= []
    i=0
    while i<len(data):
       if data["hybrid"].iloc[i]==1 or data["hybrid"].iloc[i]==-1:
         trade_index.append(i)
       i+=1
    col= [
        'Entry Index', 'Exit Index', 'Trade Duration',
        'Returns for the trade in percent', 'Type of trade']

    df = pd.DataFrame(columns=col)

    i=1
    while i<len(trade_index)-1:
        x = data['Close'].iloc[trade_index[i]]
        y = data['Close'].iloc[trade_index[i-1]]
        df.loc[i-1, 'Entry Index'] = trade_index[i-1]
        df.loc[i-1, 'Exit Index'] = trade_index[i]
        df.loc[i-1, 'Trade Duration'] = trade_index[i] - trade_index[i-1]
        df.loc[i-1, 'Returns for the trade in percent'] = ((x - y) * 100) / y

        if data["hybrid"].iloc[trade_index[i-1]]==1:
            df.loc[i-1,'Type of trade']='Long'
        else:
            df.loc[i-1,'Type of trade']='Short'

        z=data['portfolio(without stoploss)'].iloc[trade_index[i-1]]
        min=z
        j=trade_index[i-1] + 1
        while j< trade_index[i] + 1:
            if data['portfolio(without stoploss)'].iloc[j] <min:
                min = data['portfolio(without stoploss)'].iloc[j]
            j+=1
        df.loc[i-1, 'Max dip for the trade'] = ((min-z)/z)*100

        max1=data['portfolio(without stoploss)'].iloc[trade_index[i-1]]
        max2=0
        j=trade_index[i-1]
        while j<trade_index[i]+1:
            if data['portfolio(without stoploss)'].iloc[j] >max1:
                max1 = data['portfolio(without stoploss)'].iloc[j]
            t=(max1-data['portfolio(without stoploss)'].iloc[j])/max1
            if t>max2:
                max2=t
            j+=1
        df.loc[i-1, 'Max Drawdown for the trade']= max2*100
        i+=1

    df = df.reset_index(drop=True)
    return df
tradewise=tradewise(data)
tradewise

,Entry Index,Exit Index,Trade Duration,Returns for the trade in percent,Type of trade,Max dip for the trade,Max Drawdown for the trade
0,5,30,25,-41.68369,Long,-37.103501,37.103501
1,30,45,15,-0.53516,Short,-48.878062,55.815654
2,45,66,21,-7.587647,Long,-4.717996,14.649611
3,66,102,36,-15.955809,Short,-41.003629,48.396295
4,102,130,28,6.90898,Long,0.000000,8.251790
5,130,198,68,-12.683903,Short,-42.100454,49.431319
6,198,212,14,3.447808,Long,-0.225846,7.641238
7,212,236,24,-11.301382,Short,-44.345080,49.147186
8,236,248,12,-3.460202,Long,-0.778730,7.310787
9,248,265,17,2.779219,Short,-49.791816,50.896459


**METRICS**

In [104]:
def metrics_without_stoploss(data,tradewise):   ### FROM LAST ASSIGNMENT
  amt = 100000
  quant = int(amt / data['Close'].iloc[0])
  z = quant * data['Close'].iloc[-1]
  r = z - amt
  ct = 0
  i=0
  while i<len(data):
    if data["hybrid"].iloc[i] ==1:
        ct += 1
    i+=1
  print("Benchmark return =", r)
  print("Return =",tradewise['Returns for the trade in percent'].sum(),"%" )
  print("\nNumber of closed trades = ", ct)

  mean_time=tradewise['Trade Duration'].mean()
  max_time=tradewise['Trade Duration'].max()
  print("\nMaximum holding time = ",max_time,"\nMean holding time = ",(int)(mean_time))

  g_profit=data['portfolio(without stoploss)'].iloc[len(data)-1]-data['portfolio(without stoploss)'].iloc[0]
  n_profit=g_profit-(ct*20)
  print("\nGross Profit = ",g_profit,"\nNet Profit = ",n_profit)

  mean_drawdown=tradewise['Max Drawdown for the trade'].mean()
  mean_dip=tradewise['Max dip for the trade'].mean()
  max_dip=tradewise['Max dip for the trade'].min()
  max_drawdown=tradewise['Max Drawdown for the trade'].max()
  print("\nAverage drawdown for the trade = ",mean_drawdown,"%\nMaximum drawdown for the trade = ",max_drawdown,"%")
  print("\nAverage dip for the trade = ",mean_dip,"%\nMaximum dip for the trade = ",max_dip,"%")

  std_returns=tradewise['Returns for the trade in percent'].std()
  mean_returns=tradewise['Returns for the trade in percent'].mean()
  sharpe=(mean_returns/std_returns)*np.sqrt(252)
  print("Sharpe_like Ratio = ", sharpe)
metrics_without_stoploss(data,tradewise)

Benchmark return = 199150.98828125
Return = 405.2067974057555 %

Number of closed trades =  31

Maximum holding time =  165 
Mean holding time =  24

Gross Profit =  386888.50048828125 
Net Profit =  386268.50048828125

Average drawdown for the trade =  31.918615501362506 %
Maximum drawdown for the trade =  63.10546912944487 %

Average dip for the trade =  -27.214957033453192 %
Maximum dip for the trade =  -63.10546912944487 %
Sharpe_like Ratio =  2.151864333944195


**PORTFOLIO WITHOUT STOPLOSS**

In [105]:
fig=go.Figure()
fig.add_trace(go.Scatter(x=data["Date"], y=data["portfolio(without stoploss)"], line = dict(color='blue', width=2)))
fig.update_layout(title=dict(text="Portfolio value without stoploss",x=0.45,y=0.9,font=dict(size=30)),yaxis_title="portfolio value",xaxis_title="Date")
fig.show()